#### 1.原始訊息
##### 收到日期：20260813
##### 完成日期：20260817
##### 花費時間：4 Hr
需求：以加權指數自 2000/1/1 起日資料，當日收盤價若向上穿越過季線（60日），則買入；反之，則賣出。請統計出以下幾項資料：

- ⬤ 總損益、交易次數、勝率、平均賺賠比、最大連續虧損
- ⬤ 最後持有部位資訊（進場日、多空、價位）
- ⬤ （進階）統計所有出場交易之損益、持有時間

P.S：
- 1）先以金融市場習慣說法為主，查了若仍不懂請勇於提問。利用每次機會知道對方的邏輯
- 2）本策略一旦進場後，非多即空，不會空手
- 3）可能是題組，請考量日後調整彈性
- 4）不限制使用工具（個人偏好EXCEL、GOOGLE SPREADSHEET）


#### 2.策略及定義假設
- **基本設定**
  - 標的：台灣加權指數（TAIEX / Y9999）
  - 資料頻率：日資料
  - 回測起始日：2000/01/01
  - 季線：60 日簡單移動平均線（SMA60）

- **交易規則**
  - 做多
    - 昨日收盤價 ≤ 昨日 SMA60
    - 今日收盤價 > 今日 SMA60
    - 視為**向上穿越季線**，建立多頭部位（Long）
  - 做空
    - 昨日收盤價 ≥ 昨日 SMA60
    - 今日收盤價 < 今日 SMA60
    - 視為**向下穿越季線**，建立空頭部位（Short）
  - 部位規則
    - 策略開始持有部位後，維持**非多即空**
    - 不存在空手狀態
    - 出現反向訊號時：
      1. 將原有部位平倉
      2. 同時建立反向部位

- **成交假設**
  - 訊號以**當日收盤價**判斷
  - 假設以**訊號當日收盤價成交**

- **損益定義**
  - 損益先以**加權指數點數**表示
  - 暫不考慮：
    - 初始本金
    - 每點價值
    - 交易成本
    - 手續費
    - 稅費
    - 滑價
      - 原本預期成交的價格，和實際真正成交的價格之間的差距。
      - 例如你看到加權指數在 20,000 點出現買進訊號，理論上希望用 20,000 點成交，但實際下單後可能成交在 20,005 點，這多出來的 5 點就是滑價。
      - 滑價常見原因包括市場快速波動、流動性不足、買賣價差，以及從訊號產生到訂單真正進市場之間的時間差。

- **交易定義**
  - 一筆完整交易定義為：
    - **進場 → 持有 → 出場**
  - 反手時：
    - 原部位完成一筆交易
    - 同時開始下一筆反向交易
  - 尚未出場的最後一筆部位視為**未平倉部位（Open Position）**

- **持有時間**
  - 持有時間以**交易日（Trading Days）**計算
  - 定義：
    - 今日收盤進場，下一個交易日收盤出場，持有時間為 1 個交易日
  - 不使用日曆日計算，因此週末及休市日不額外計入持有時間

- **績效指標**
  - 總損益
    - 所有已完成交易之損益點數加總
  - 交易次數
    - 完成「進場 → 出場」的交易筆數
    - 未平倉的最後部位不列入已完成交易次數
  - 勝率
    - 獲利交易次數占全部已完成交易次數的比例
  - 平均賺賠比
    - 平均獲利交易的獲利點數，相對於平均虧損交易虧損點數絕對值的比例
  - 最大連續虧損
    - 歷史交易紀錄中，連續出現虧損交易的最大筆數

- **最後持有部位**
  - 紀錄回測資料截止日仍持有之部位：
    - 進場日
    - 多空方向（Long / Short）
    - 進場價位

- **出場交易明細**
  - 每筆已完成交易至少紀錄：
    - 進場日
    - 出場日
    - 多空方向
    - 進場價
    - 出場價
    - 損益點數
    - 持有交易日數

- **第一筆部位如何建立**
  - 等待回測起始日後第一次穿越訊號再進場

##### 3.程式內容

DB 連線測試

In [29]:
import pandas as pd
import os
from dotenv import load_dotenv
import psycopg

load_dotenv("../../.env")

try:
    conn = psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        connect_timeout=5
    )

    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                current_database(),
                current_user,
                version();
        """)

        database, user, version = cur.fetchone()

    print("PostgreSQL 連線成功")
    print("Database :", database)
    print("User     :", user)
    print("Version  :", version)

except psycopg.Error as e:
    print("PostgreSQL 連線失敗")
    print(e)

finally:
    if 'conn' in locals():
        conn.close()

PostgreSQL 連線成功
Database : market_data
User     : market_data_user
Version  : PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [30]:
if 'conn' not in globals() or conn.closed:
    conn = psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        connect_timeout=5
    )

print("PostgreSQL 連線狀態：", "已連線" if not conn.closed else "已關閉")

PostgreSQL 連線狀態： 已連線


In [31]:
conn.rollback()

In [32]:
import pandas as pd

sql = """
SELECT
    md.trade_date,
    md.close_price
    
FROM instrument i
JOIN market_daily md
    ON md.instrument_id = i.id
WHERE i.code = %s
ORDER BY md.trade_date;
"""

with conn.cursor() as cur:
    cur.execute(sql, ("Y9999",))

    rows = cur.fetchall()
    columns = [desc.name for desc in cur.description]

df = pd.DataFrame(rows, columns=columns)
df["close_price"] = pd.to_numeric(df["close_price"])
df["trade_date"] = pd.to_datetime(df["trade_date"])

In [33]:
df.head()

,trade_date,close_price
0,1999-06-01,7397.62
1,1999-06-02,7488.03
2,1999-06-03,7572.91
3,1999-06-04,7590.44
4,1999-06-05,7639.30


In [34]:
df[["trade_date", "close_price"]]

,trade_date,close_price
0,1999-06-01,7397.62
1,1999-06-02,7488.03
2,1999-06-03,7572.91
3,1999-06-04,7590.44
4,1999-06-05,7639.30
5,1999-06-07,7802.69
6,1999-06-08,7892.13
7,1999-06-09,7957.71
8,1999-06-10,7996.76
9,1999-06-11,7979.40


In [35]:
# df.describe()
# df.head()
df.tail()
# df.info()
# df.shape


,trade_date,close_price
6728,2026-08-06,44396.70
6729,2026-08-07,44225.91
6730,2026-08-10,44928.76
6731,2026-08-11,45120.72
6732,2026-08-12,45518.07


In [36]:
df.iloc[2]
df["test"] = 1
df.loc[0, "test"] = 10000
df.loc[61]

trade_date     1999-08-20 00:00:00
close_price                8117.42
test                             1
Name: 61, dtype: object

sma 

In [37]:
def get_sma(index, dateCount = 60):
    sumValue = 0
    for i in range (dateCount):
        currIndex = index - i
        currCloseValue = df.loc[currIndex,"close_price"]
        sumValue += currCloseValue
    return sumValue/dateCount

sma = get_sma(100)
smaa = df["close_price"].rolling(60).mean().iloc[100]


In [38]:
df["sma60"] = df["close_price"].rolling(60).mean()

signal : Enum
- "Long"
- "Short"
- "-"

In [39]:

def get_signal(index):
    currValue = df.loc[index, "close_price"]
    currSma = df.loc[index, "sma60"]

    yestValue = df.loc[index - 1, "close_price"]
    yestSma = df.loc[index - 1, "sma60"]

    if currValue > currSma and yestValue <= yestSma:
        # print(index, "L")
        return "Long"

    elif currValue < currSma and yestValue >= yestSma:
        # print(index, "S")
        return "Short"
    else:
        return "-"

df["signal"] = "-"
for index in  range(60, len(df)):
    
    df.loc[index, "signal"] = get_signal(index)
        
    

In [40]:
df.tail()

,trade_date,close_price,test,sma60,signal
6728,2026-08-06,44396.70,1,44231.234833,-
6729,2026-08-07,44225.91,1,44278.758333,Short
6730,2026-08-10,44928.76,1,44331.708500,Long
6731,2026-08-11,45120.72,1,44397.514500,-
6732,2026-08-12,45518.07,1,44474.618667,-


In [41]:
signal_df = df.loc[
    (df["signal"] != "-") & (df["trade_date"] >= pd.Timestamp("2000-01-01")),
    ["trade_date", "close_price", "sma60", "signal"]
].copy()
signal_df = signal_df.reset_index(drop=True)
signal_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 334 entries, 0 to 333
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype        
---  ------       --------------  -----        
 0   trade_date   334 non-null    datetime64[s]
 1   close_price  334 non-null    float64      
 2   sma60        334 non-null    float64      
 3   signal       334 non-null    str          
dtypes: datetime64[s](1), float64(2), str(1)
memory usage: 10.6 KB


交易次數

In [42]:
print("交易次數",len(signal_df))

交易次數 334


In [43]:
def get_profit(index):
    currValue = signal_df.loc[index, "close_price"]
    yestValue = signal_df.loc[index - 1, "close_price"]
    isLong = signal_df.loc[index -1, "signal"] == "Long"
    
    if(isLong):
        return currValue - yestValue
    else: 
        return yestValue - currValue
    
def get_hold_day(index):
    currDay = signal_df.loc[index, "trade_date"]
    yestDay = signal_df.loc[index - 1, "trade_date"]
    hold_day = (currDay - yestDay).days
    return hold_day



for index in  range(1, len(signal_df)):
    signal_df.loc[index - 1, "profit"] = get_profit(index)
    signal_df.loc[index -1, "hold_day"] = get_hold_day(index)

In [44]:
signal_df.tail()

,trade_date,close_price,sma60,signal,profit,hold_day
329,2026-07-21,44232.87,43709.963000,Long,-578.03,3.0
330,2026-07-24,43654.84,43964.307500,Short,-956.76,12.0
331,2026-08-05,44611.60,44189.595167,Long,-385.69,2.0
332,2026-08-07,44225.91,44278.758333,Short,-702.85,3.0
333,2026-08-10,44928.76,44331.708500,Long,NaN,NaN


In [45]:
sum_profit = 0
win_count = 0
loss_count = 0

win_profit_sum = 0
loss_profit_sum = 0


loss_continue_max = 0
curr_loss_continue_count = 0

def set_loss_coutinue(is_profit_win):
    global curr_loss_continue_count
    global loss_continue_max
    if(is_profit_win):
        
        if(curr_loss_continue_count > loss_continue_max):
            loss_continue_max = curr_loss_continue_count
        curr_loss_continue_count = 0
    else:
        curr_loss_continue_count = curr_loss_continue_count + 1
    

for index in range(0, len(signal_df) - 1):
    sum_profit = sum_profit+signal_df.loc[index, "profit"]
    profit = signal_df.loc[index,"profit"]
    
    if(profit > 0):
        win_count = win_count + 1
        win_profit_sum = win_profit_sum + profit
        set_loss_coutinue(True)
        
    else:
        set_loss_coutinue(False)
        loss_count = loss_count + 1
        loss_profit_sum = loss_profit_sum + profit

total_count = win_count + loss_count
if total_count > 0:
    win_rate = win_count / total_count
else:
    win_rate = 0
    
print("sum_profit:",f"{sum_profit:.2f}")
print("win_ratef:",f"{win_rate*100:.2f}%")

win_profit_mean = win_profit_sum / win_count
loss_profit_mean = loss_profit_sum / loss_count

print ("win_profit_mean:", f"{win_profit_mean:.2f}")
print ("loss_profit_mean:", f"{loss_profit_mean:.2f}")
profit_ratio = abs(win_profit_mean) / abs(loss_profit_mean)
print ("profit_ratio:", f"{profit_ratio:.2f}")
print ("loss_continue_max:",loss_continue_max)



sum_profit: 32879.93
win_ratef: 23.42%
win_profit_mean: 921.20
loss_profit_mean: -152.84
profit_ratio: 6.03
loss_continue_max: 24


In [46]:
signal_df.iloc[len(signal_df)-1]


trade_date     2026-08-10 00:00:00
close_price               44928.76
sma60                   44331.7085
signal                        Long
profit                         NaN
hold_day                       NaN
Name: 333, dtype: object

In [47]:
print("-" * 10, "統計資訊", "-" * 10)
print("總損益",f"{sum_profit:.2f}")
print("交易次數",len(signal_df))
print("勝率",f"{win_rate*100:.2f}%")
print("平均賺賠比", f"{profit_ratio:.2f}")
print("最大連續虧損", loss_continue_max)
print("-" * 10, "最後持有部位資訊", "-" * 10)

print("進場日", signal_df.loc[len(signal_df)-1, "trade_date"])
print("多空", signal_df.loc[len(signal_df)-1, "signal"])
print("價位", signal_df.loc[len(signal_df)-1, "close_price"])

print("-" * 10, "統計所有出場交易之損益、持有時間", "-" * 10)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

print(signal_df)
print("-" * 10, "其它資訊", "-" * 10)
print("最高持有日",signal_df["hold_day"].max())
print("單次最高獲利",f"{signal_df["profit"].max():.2f}")
print("單次最高虧損",f"{signal_df["profit"].min():.2f}")

---------- 統計資訊 ----------
總損益 32879.93
交易次數 334
勝率 23.42%
平均賺賠比 6.03
最大連續虧損 24
---------- 最後持有部位資訊 ----------
進場日 2026-08-10 00:00:00
多空 Long
價位 44928.76
---------- 統計所有出場交易之損益、持有時間 ----------
    trade_date  close_price         sma60 signal   profit  hold_day
0   2000-03-13      8811.95   9190.846500  Short  -721.92      10.0
1   2000-03-23      9533.87   9330.423000   Long  -159.26      22.0
2   2000-04-14      9374.61   9602.798500  Short   259.14      54.0
3   2000-06-07      9115.47   9088.918167   Long   -47.59       1.0
4   2000-06-08      9067.88   9089.974833  Short  3691.76     215.0
5   2001-01-09      5376.12   5344.077000   Long   231.61      83.0
6   2001-04-02      5607.73   5653.865167  Short  1527.22     217.0
7   2001-11-05      4080.51   4068.742000   Long  1787.32     178.0
8   2002-05-02      5867.83   6049.507333  Short  1277.95     174.0
9   2002-10-23      4589.88   4538.422333   Long   -91.15       7.0
10  2002-10-30      4498.73   4511.314333  Short   -80.41 

In [48]:


def set_single_max(index):
    last_signal_date = signal_df.loc[index -1 , "trade_date"]
    curr_signal_date = signal_df.loc[index , "trade_date"]
    
    max_period_price = df.loc[
        (df["trade_date"] >= last_signal_date) &
        (df["trade_date"] <= curr_signal_date),
        "close_price"
    ].max()
    min_period_price = df.loc[
            (df["trade_date"] >= last_signal_date) &
            (df["trade_date"] <= curr_signal_date),
        "close_price"
    ].min()
    
    max_period_profit = max_period_price - signal_df.loc[index-1,"close_price"]
    min_period_profit = min_period_price - signal_df.loc[index-1,"close_price"] 
    
    
    signal_df.loc[index-1, "max_period_price"] = max_period_price
    signal_df.loc[index-1, "min_period_price"] = min_period_price
    
    signal_df.loc[index-1, "max_period_profit"] = max_period_profit
    signal_df.loc[index-1, "min_period_profit"] = min_period_profit
    

for index in range(1, len(signal_df)):
    set_single_max(index)



In [49]:
signal_df

,trade_date,close_price,sma60,signal,profit,hold_day,max_period_price,min_period_price,max_period_profit,min_period_profit
0,2000-03-13,8811.95,9190.846500,Short,-721.92,10.0,9533.87,8536.05,721.92,-275.90
1,2000-03-23,9533.87,9330.423000,Long,-159.26,22.0,10186.17,9374.61,652.30,-159.26
2,2000-04-14,9374.61,9602.798500,Short,259.14,54.0,9374.61,8349.91,0.00,-1024.70
3,2000-06-07,9115.47,9088.918167,Long,-47.59,1.0,9115.47,9067.88,0.00,-47.59
4,2000-06-08,9067.88,9089.974833,Short,3691.76,215.0,9067.88,4614.63,0.00,-4453.25
5,2001-01-09,5376.12,5344.077000,Long,231.61,83.0,6104.24,5339.40,728.12,-36.72
6,2001-04-02,5607.73,5653.865167,Short,1527.22,217.0,5608.50,3446.26,0.77,-2161.47
7,2001-11-05,4080.51,4068.742000,Long,1787.32,178.0,6462.30,4080.51,2381.79,0.00
8,2002-05-02,5867.83,6049.507333,Short,1277.95,174.0,5910.69,3850.04,42.86,-2017.79
9,2002-10-23,4589.88,4538.422333,Long,-91.15,7.0,4601.37,4498.73,11.49,-91.15
